# Resolving Conflicts with Precedence Declarations

The notebook [02-Conflicts-Resolved.ipynb](02-Conflicts-Resolved.ipynb) has resolved the conflicts of the ambiguous
grammar for arithmetical expressions by encoding the precedences of the operators into the grammar.  The parser
generators of the `yacc` family, among them `Bison` and `Ply`, take a different approach:  They keep the ambiguous
grammar and resolve its *shift/reduce conflicts* with *precedence declarations*.  `Larkita` supports the precedence
declarations of `Bison`.

The grammar below extends the ambiguous grammar of the notebook [01-Conflicts.ipynb](01-Conflicts.ipynb) by the
comparison operators `<` and `==` and by the unary minus.

## Precedence Declarations

* Every declaration consists of one of the keywords `%left`, `%right`, `%nonassoc`, or `%precedence`, followed by a
  list of literals or terminals.  The symbols of a declaration have the same *precedence*, which is higher than
  the precedence of the symbols that are declared above them.
* `%left` declares **left associative** operators, `%right` declares **right associative** operators, and
  `%nonassoc` declares **non-associative** operators:  `1 < 2 < 3` is a syntax error.  `%precedence` declares a
  precedence without an associativity.
* A grammar rule has the precedence of the last terminal on its right hand side.  The annotation `%prec UMINUS`
  gives the rule for the unary minus the precedence of `UMINUS` instead, so that the unary minus binds stronger
  than `*` and `/`, but weaker than `^`.  `UMINUS` is not a terminal:  It is only a name for a precedence.  The
  annotation `%prec` has to precede an alias or an annotation `%ignore`.

| precedence | operators   | associativity   |
|:-----------|:------------|:----------------|
| 1          | `<`, `==`   | non-associative |
| 2          | `+`, `-`    | left            |
| 3          | `*`, `/`    | left            |
| 4          | unary `-`   | &mdash;         |
| 5          | `^`         | right           |

In [ ]:
from IPython.display import Code, display

Code(filename='precedence.lark', language='larkita')

## Resolving the Conflicts

If the parser could either reduce with a rule or shift a token, and both the rule and the token have a precedence,
`Larkita` resolves this shift/reduce conflict in the same way as `Bison`:

1. If the token has a higher precedence than the rule, the token is shifted.
2. If the rule has a higher precedence than the token, the parser reduces with the rule.
3. If both have the same precedence, the associativity decides:  `%left` means *reduce*, `%right` means *shift*,
   and `%nonassoc` turns the token into a syntax error.  As `%precedence` declares no associativity, the conflict
   is not resolved.

All other shift/reduce conflicts are resolved in favour of shifting and reported with a `ConflictWarning`.  We turn
this warning into an exception.  As no exception is raised below, all conflicts have been resolved by the
precedence declarations.

In [ ]:
import warnings

from larkita import Larkita, ConflictWarning

with warnings.catch_warnings():
    warnings.simplefilter('error', ConflictWarning)
    parser = Larkita.open('precedence.lark', out_file='precedence.out')
print(len(parser.conflicts), 'conflicts')

## Inspecting the Output File

The section `Precedence` of the output file lists the precedence declarations together with the precedences of
the rules.

In [ ]:
with open('precedence.out') as handle:
    report = handle.read()
print(report[report.index('Precedence\n'):report.index('Conflicts\n')].rstrip())

Every conflict is listed together with the reason for its resolution, which is written in the same way as by
`Bison`:  `("+" < "*")` says that `"+"` has a lower precedence than `"*"`, while `(%left "+")` says that the rule
and the token have the same precedence and that this precedence is left associative.  The following cell prints the
state that the parser reaches after it has read an expression of the form `expr "+" expr`.

In [ ]:
states = report[report.index('States\n'):].split('\n\n\n')
print(next(state for state in states if 'expr → expr "+" expr •' in state))

In this state, the parser shifts the tokens `*`, `/`, and `^`, since they have a higher precedence than the rule
`expr → expr "+" expr`, which has the precedence of `"+"`.  The tokens `+` and `-` have the same precedence as this
rule.  As they are left associative, the parser reduces.  Finally, the tokens `<` and `==` have a lower precedence
than the rule, so the parser reduces, too.

The canonical LR(1) automaton contains the states for expressions twice:  once for expressions at the top level
and once for expressions inside parentheses, whose look-ahead sets contain `")"` instead of `$END`.  Hence, every
conflict occurs twice.

## Building an Abstract Syntax Tree

The parser generated by `Larkita` returns the abstract syntax tree as a nested tuple.  Hence we can use the function
`tuple2dot` directly in order to display it.

In [ ]:
%run ../AST2Dot.ipynb

In [ ]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:  # declarations for the type checker, never executed
    import graphviz as gv
    type AST = str | int | float | tuple[str, *tuple["AST", ...]]
    def tuple2dot(t: AST) -> gv.Digraph: ...

The function `test(s)` takes a string `s` as its argument and tries to parse this string.  If all goes well,
an abstract syntax tree is returned and displayed.  If the string can't be parsed, `Larkita` raises an exception
of class `UnexpectedInput`, whose message is printed.

In [ ]:
from larkita import UnexpectedInput

def test(s):
    try:
        t = parser.parse(s)
    except UnexpectedInput as e:
        print(f'Syntax error:\n{e}')
        return None
    d = tuple2dot(t)
    display(d)
    return t

In [ ]:
test('1 + 2 * 3')

In [ ]:
test('1 + 2 < 3 * 4')

The next two examples check the associativities:  the additive operators group to the left, while the
exponentiation operator groups to the right.

In [ ]:
test('1 - 2 - 3')

In [ ]:
test('2 ^ 3 ^ 4')

The unary minus binds stronger than `*`, but weaker than `^`.

In [ ]:
test('-2 * 3')

In [ ]:
test('-2 ^ 2')

Finally, the comparison operators are non-associative.  Hence the following string cannot be parsed:  The
second `<` is a syntax error.

In [ ]:
test('1 < 2 < 3')